# data cleaning and composite features
Checking distributions, dropping noisy columns, and creating composite scores.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

### Load data

In [2]:
df = pd.read_excel('mental_health.xlsx')
target = 'Has_Mental_Health_Issue'

c0 = df[df[target] == 0]
c1 = df[df[target] == 1]
print("Healthy:", len(c0), "At Risk:", len(c1))

Healthy: 784 At Risk: 9216


### Compare averages of lifestyle columns

In [3]:
print("Sleep healthy:", round(c0['Sleep_Hours_Night'].mean(), 2))
print("Sleep at risk:", round(c1['Sleep_Hours_Night'].mean(), 2))

print("Screen time healthy:", round(c0['Screen_Time_Hours_Day'].mean(), 2))
print("Screen time at risk:", round(c1['Screen_Time_Hours_Day'].mean(), 2))

print("Age healthy:", round(c0['Age'].mean(), 2))
print("Age at risk:", round(c1['Age'].mean(), 2))

Sleep healthy: 6.81
Sleep at risk: 6.82
Screen time healthy: 7.08
Screen time at risk: 7.06
Age healthy: 34.89
Age at risk: 34.95


### Filter out non-significant features (p >= 0.05)

In [4]:
significant_cols = []

for col in df.columns:
    if col == target:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        _, p_val = stats.ttest_ind(c0[col].dropna(), c1[col].dropna(), equal_var=False)
    else:
        _, p_val, _, _ = stats.chi2_contingency(pd.crosstab(df[col], df[target]))
    if p_val < 0.05:
        significant_cols.append(col)

print("Original features:", len(df.columns) - 1)
print("Significant features:", len(significant_cols))

Original features: 50
Significant features: 23


### Create composite scores to combine symptoms

In [5]:
symptoms = [
    'Work_Stress_Level', 'Financial_Stress', 'Feeling_Sad_Down',
    'Anxious_Nervous', 'Loneliness', 'Fatigue', 'Poor_Appetite_Or_Overeating',
    'Feeling_Worthless', 'Irritability', 'Compulsive_Behavior',
    'Obsessive_Thoughts', 'Concentration_Difficulty', 'Sleep_Trouble'
]

# Total distress score
df['Total_Distress_Score'] = df[symptoms].sum(axis=1) - df['Social_Support'] - df['Job_Satisfaction']

# Severe clinical risk score
df['Severe_Risk_Score'] = (
    df['Suicidal_Thoughts'] * 3 +
    df['Self_Harm_Thoughts'] * 3 +
    df['Family_History_Mental_Illness'] * 2 +
    df['Trauma_History'] * 2 +
    df['On_Medication']
)

# Combined overall risk index
df['Overall_Risk_Index'] = df['Total_Distress_Score'] + (df['Severe_Risk_Score'] * 5)

print("Overall Risk Index Healthy Mean:", round(df[df[target]==0]['Overall_Risk_Index'].mean(), 2))
print("Overall Risk Index Risk Mean   :", round(df[df[target]==1]['Overall_Risk_Index'].mean(), 2))

Overall Risk Index Healthy Mean: 55.92
Overall Risk Index Risk Mean   : 65.07


### Save cleaned composite dataset

In [6]:
clean_cols = ['Overall_Risk_Index', 'Total_Distress_Score', 'Severe_Risk_Score', 'Social_Support', 'Has_Mental_Health_Issue']
df_clean = df[clean_cols]

df_clean.to_csv('mental_health_clean.csv', index=False)
print("Saved mental_health_clean.csv, shape:", df_clean.shape)
df_clean.head()

Saved mental_health_clean.csv, shape: (10000, 5)


,Overall_Risk_Index,Total_Distress_Score,Severe_Risk_Score,Social_Support,Has_Mental_Health_Issue
0,63,53,2,9,1
1,71,71,0,3,1
2,84,74,2,5,1
3,51,41,2,1,1
4,78,78,0,1,1
